In [2]:
import sys, time, json
sys.path.insert(0, "..")
from src.spark_utils import get_spark
from pyspark.sql import functions as F

spark = get_spark("04-models", driver_memory="6g")
df = spark.read.parquet("../data/yellow_2016_03_features.parquet")
print(f"Loaded: {df.count():,} rows")

# Reproducible 80/20 split
train, test = df.randomSplit([0.8, 0.2], seed=42)
train.cache()
test.cache()
print(f"Train: {train.count():,}  Test: {test.count():,}")

Loaded: 7,937,686 rows
Train: 6,350,117  Test: 1,587,569


In [3]:
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
)

categorical = ["VendorID", "RatecodeID", "pickup_hour", "pickup_dayofweek"]
numeric = [
    "passenger_count", "trip_distance", "fare_amount",
    "extra", "mta_tax", "tolls_amount",
    "trip_duration_min", "avg_speed_mph", "pickup_day",
    "is_weekend", "is_rush_hour", "is_night",
]

# Index then one-hot each categorical
indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in categorical
]
encoders = [
    OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_oh")
    for c in categorical
]

# Combine numeric + one-hot vectors into one feature vector
assembler = VectorAssembler(
    inputCols=numeric + [f"{c}_oh" for c in categorical],
    outputCol="features_raw",
)

# Scaling matters for LR, harmless for GBT
scaler = StandardScaler(
    inputCol="features_raw", outputCol="features",
    withMean=False, withStd=True,
)

preprocessing_stages = indexers + encoders + [assembler, scaler]
print(f"Preprocessing stages: {len(preprocessing_stages)}")

Preprocessing stages: 10


In [4]:
from pyspark.ml import Pipeline
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(
    featuresCol="features",
    labelCol="tip_amount",
    regParam=0.1,
    elasticNetParam=0.0,
    maxIter=50,
)

lr_pipeline = Pipeline(stages=preprocessing_stages + [lr])

t0 = time.time()
lr_model = lr_pipeline.fit(train)
lr_train_time = time.time() - t0
print(f"LR training time: {lr_train_time:.1f}s")

LR training time: 19.4s


In [5]:
from pyspark.ml.evaluation import RegressionEvaluator

preds = lr_model.transform(test)

def evaluate(predictions, label_col="tip_amount"):
    metrics = {}
    for m in ["rmse", "mae", "r2"]:
        ev = RegressionEvaluator(
            labelCol=label_col, predictionCol="prediction", metricName=m
        )
        metrics[m] = ev.evaluate(predictions)
    return metrics

lr_metrics = evaluate(preds)
print(f"LR  RMSE: {lr_metrics['rmse']:.4f}")
print(f"LR  MAE:  {lr_metrics['mae']:.4f}")
print(f"LR  R²:   {lr_metrics['r2']:.4f}")

LR  RMSE: 1.2971
LR  MAE:  0.7056
LR  R²:   0.6857


In [6]:
import os
os.makedirs("../results", exist_ok=True)

results = {
    "linear_regression": {
        **lr_metrics,
        "train_time_sec": round(lr_train_time, 2),
        "train_rows": train.count(),
    }
}

with open("../results/model_metrics.json", "w") as f:
    json.dump(results, f, indent=2)
print(json.dumps(results, indent=2))

{
  "linear_regression": {
    "rmse": 1.2971336041244856,
    "mae": 0.7056452123944453,
    "r2": 0.685679484531202,
    "train_time_sec": 19.42,
    "train_rows": 6350117
  }
}


In [ ]:
from pyspark.ml.regression import GBTRegressor

gbt = GBTRegressor(
    featuresCol="features",
    labelCol="tip_amount",
    maxIter=50, # number of trees
    maxDepth=6,
    stepSize=0.1,
    seed=42,
)

gbt_pipeline = Pipeline(stages=preprocessing_stages + [gbt])

t0 = time.time()
gbt_model = gbt_pipeline.fit(train)
gbt_train_time = time.time() - t0
print(f"GBT training time: {gbt_train_time:.1f}s")

GBT training time: 212.0s


In [8]:
gbt_preds = gbt_model.transform(test)
gbt_metrics = evaluate(gbt_preds)
print(f"GBT RMSE: {gbt_metrics['rmse']:.4f}")
print(f"GBT MAE:  {gbt_metrics['mae']:.4f}")
print(f"GBT R²:   {gbt_metrics['r2']:.4f}")

GBT RMSE: 1.3029
GBT MAE:  0.6977
GBT R²:   0.6829


In [9]:
results["gbt"] = {
    **gbt_metrics,
    "train_time_sec": round(gbt_train_time, 2),
    "train_rows": train.count(),
}

with open("../results/model_metrics.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"{'Metric':<8} {'LR':>10} {'GBT':>10} {'Δ':>10}")
print("-" * 42)
for m in ["rmse", "mae", "r2"]:
    lr_v = results["linear_regression"][m]
    gb_v = results["gbt"][m]
    delta = gb_v - lr_v
    sign = "↓" if (m != "r2" and delta < 0) or (m == "r2" and delta > 0) else "↑"
    print(f"{m.upper():<8} {lr_v:>10.4f} {gb_v:>10.4f} {delta:>+10.4f} {sign}")

print(f"{'time':<8} {results['linear_regression']['train_time_sec']:>10.1f} "
      f"{results['gbt']['train_time_sec']:>10.1f}")

Metric           LR        GBT          Δ
------------------------------------------
RMSE         1.2971     1.3029    +0.0058 ↑
MAE          0.7056     0.6977    -0.0079 ↓
R2           0.6857     0.6829    -0.0028 ↑
time           19.4      212.0


In [ ]:
import pandas as pd

gbt_stage = gbt_model.stages[-1]
importances = gbt_stage.featureImportances.toArray()

sample = gbt_model.transform(test.limit(1))

# Try features_raw first (assembler output, retains metadata), fall back to features and dump structure if both fail.
def get_feature_names(df_with_features):
    for col in ("features_raw", "features"):
        if col not in df_with_features.columns:
            continue
        meta = df_with_features.schema[col].metadata.get("ml_attr", {})
        attrs = meta.get("attrs")
        num_attrs = meta.get("num_attrs")
        if attrs and num_attrs:
            names = [None] * num_attrs
            for _, attr_list in attrs.items():
                for a in attr_list:
                    names[a["idx"]] = a["name"]
            return names
    return None

feature_names = get_feature_names(sample)

if feature_names is None:
    # Diagnostic, show what's actually in the metadata
    print("Could not auto-extract names. Metadata seen:")
    for col in ("features_raw", "features"):
        if col in sample.columns:
            print(f"  {col}: {sample.schema[col].metadata}")
    raise RuntimeError("Stop here, paste the metadata above.")

print(f"Importances: {len(importances)}, Names: {len(feature_names)}")

imp_df = pd.DataFrame({"feature": feature_names, "importance": importances})
imp_df = imp_df.sort_values("importance", ascending=False)

def parent(name):
    for cat in ["VendorID", "RatecodeID", "pickup_hour", "pickup_dayofweek"]:
        if name.startswith(cat):
            return cat
    return name

imp_df["parent"] = imp_df["feature"].apply(parent)
agg_df = (imp_df.groupby("parent")["importance"]
                .sum().reset_index()
                .sort_values("importance", ascending=False))

print("\n=== Aggregated ===")
print(agg_df.to_string(index=False))

print("\n=== Per-slot top 15 ===")
print(imp_df[["feature", "importance"]].head(15).to_string(index=False))

imp_df.to_csv("../results/gbt_feature_importance_perslot.csv", index=False)
agg_df.to_csv("../results/gbt_feature_importance.csv", index=False)

Importances: 52, Names: 52

=== Aggregated ===
           parent  importance
      fare_amount    0.858187
     tolls_amount    0.032176
    avg_speed_mph    0.027952
      pickup_hour    0.020103
    trip_distance    0.012119
       RatecodeID    0.011681
trip_duration_min    0.008435
       pickup_day    0.006133
            extra    0.004957
  passenger_count    0.004079
 pickup_dayofweek    0.004011
          mta_tax    0.003384
       is_weekend    0.002836
         VendorID    0.001835
     is_rush_hour    0.001327
         is_night    0.000784

=== Per-slot top 15 ===
              feature  importance
          fare_amount    0.858187
         tolls_amount    0.032176
        avg_speed_mph    0.027952
        trip_distance    0.012119
    trip_duration_min    0.008435
           pickup_day    0.006133
                extra    0.004957
      RatecodeID_oh_2    0.004646
    pickup_hour_oh_13    0.004446
      passenger_count    0.004079
              mta_tax    0.003384
      Rate

In [11]:
spark.stop()